<a href="https://colab.research.google.com/github/ayush121325/Agro-weather-AI/blob/main/03_irrigation_advisory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUT_DIR = "/content/drive/MyDrive/Avishkar_Hackathon/processed"

code = '''
def get_irrigation_decision(
    rainfall_deviation_pct,
    predicted_yield_current,
    predicted_yield_normal,
    dry_spell_days,
    soil_adequacy_index,
    crop_name="Rice"
):
    """
    Converts weather deficit + ML yield-drop risk into a 3-tier advisory:
    'Safe', 'Monitor', or 'Irrigate Now'.
    """
    # 1. Calculate % yield drop caused by current rainfall deficit
    if predicted_yield_normal > 0:
        yield_drop_pct = ((predicted_yield_normal - predicted_yield_current) / predicted_yield_normal) * 100.0
    else:
        yield_drop_pct = 0.0

    # 2. High water-sensitivity crops need earlier protection
    water_sensitive_crops = {"rice", "sugarcane", "banana", "potato", "cotton(lint)", "wheat"}
    is_sensitive = str(crop_name).strip().lower() in water_sensitive_crops

    # Adjust thresholds slightly if soil adequacy is poor (< 0.5)
    deficit_trigger = -18.0 if (is_sensitive or soil_adequacy_index < 0.5) else -22.0
    dry_spell_trigger = 12 if is_sensitive else 15

    # 3. Decision Logic (IMD Deficient = <= -20% rainfall deviation)
    if (rainfall_deviation_pct <= deficit_trigger) and (yield_drop_pct >= 10.0 or dry_spell_days >= dry_spell_trigger):
        status = "Irrigate Now"
        reason = (
            f"Genuine yield-threatening deficit: rainfall is {rainfall_deviation_pct:.1f}% below normal, "
            f"predicted yield drop is {yield_drop_pct:.1f}%, dry spell is {dry_spell_days} days."
        )
    elif (rainfall_deviation_pct <= -10.0) or (dry_spell_days >= 7 and yield_drop_pct >= 5.0):
        status = "Monitor"
        reason = (
            f"Moderate moisture watch: rainfall deviation {rainfall_deviation_pct:.1f}%, "
            f"predicted yield risk {yield_drop_pct:.1f}%. Hold irrigation and re-check in 3 days."
        )
    else:
        status = "Safe"
        reason = (
            f"Normal variability: rainfall deviation ({rainfall_deviation_pct:.1f}%) and "
            f"yield risk ({yield_drop_pct:.1f}%) are within safe bounds. No irrigation needed."
        )

    return {
        "decision": status,
        "yield_drop_pct": round(yield_drop_pct, 2),
        "reason": reason
    }
'''

with open(f"{OUT_DIR}/irrigation_advisory.py", "w") as f:
    f.write(code)

print("Saved irrigation_advisory.py to:", OUT_DIR)

Mounted at /content/drive
Saved irrigation_advisory.py to: /content/drive/MyDrive/Avishkar_Hackathon/processed


In [ ]:
import sys
sys.path.append(OUT_DIR)
from irrigation_advisory import get_irrigation_decision

scenarios = [
    ("Routine 5-day dry spell (should be Safe)", -6.0, 2.45, 2.50, 5, 0.75, "Rice"),
    ("Early monsoon delay (should be Monitor)", -15.0, 2.30, 2.50, 9, 0.65, "Wheat"),
    ("Severe Bihar Kharif deficit (should be Irrigate Now)", -32.0, 1.85, 2.50, 16, 0.45, "Rice"),
]

for title, dev, y_cur, y_norm, dry, soil_idx, crop in scenarios:
    res = get_irrigation_decision(dev, y_cur, y_norm, dry, soil_idx, crop)
    print(f"\n[{title}]")
    print(" -> Decision:", res["decision"])
    print(" -> Reason:  ", res["reason"])


[Routine 5-day dry spell (should be Safe)]
 -> Decision: Safe
 -> Reason:   Normal variability: rainfall deviation (-6.0%) and yield risk (2.0%) are within safe bounds. No irrigation needed.

[Early monsoon delay (should be Monitor)]
 -> Decision: Monitor
 -> Reason:   Moderate moisture watch: rainfall deviation -15.0%, predicted yield risk 8.0%. Hold irrigation and re-check in 3 days.

[Severe Bihar Kharif deficit (should be Irrigate Now)]
 -> Decision: Irrigate Now
 -> Reason:   Genuine yield-threatening deficit: rainfall is -32.0% below normal, predicted yield drop is 26.0%, dry spell is 16 days.
